In [1]:
import os, random, warnings
import numpy as np
import pandas as pd
import librosa
import librosa.display
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

warnings.filterwarnings("ignore")

In [2]:


# ── 2. CONFIGURATION ─────────────────────────────────────────
class CFG:
    _base      = "/kaggle/input/competitions/digitrecognition-ee708"
    train_csv  = f"{_base}/train.csv"
    test_csv   = f"{_base}/sample_submission.csv"
    train_dir  = f"{_base}/train_audio/train_audio"
    test_dir   = f"{_base}/test_audio/test_audio"

    # precomputed spec cache dirs
    train_spec_dir = "/kaggle/working/train_specs"
    test_spec_dir  = "/kaggle/working/test_specs"

    # audio
    sample_rate = 16000
    duration    = 1.0
    n_mels      = 128
    n_fft       = 512
    hop_length  = 160
    fmin        = 50
    fmax        = 8000

    # training  ← KEY CHANGES FOR SPEED
    n_folds     = 5
    epochs      = 35 # increased - mod4
    batch_size  = 256      # was 128 — T4 handles 256 fine, ~2x throughput
    lr          = 3e-4
    weight_decay= 1e-4
    num_workers = 2        # was 4 — lower avoids DataLoader deadlocks on Kaggle
    seed        = 42
    device      = "cuda" if torch.cuda.is_available() else "cpu"
    mixed_prec  = True
    tta_rounds  = 5        # was 5 — saves ~25 min total

def set_seed(seed=CFG.seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
   

In [3]:
# ── 3. FEATURE EXTRACTION ────────────────────────────────────
def load_audio(path, sr=CFG.sample_rate, duration=CFG.duration):
    """Load wav, resample, pad/trim to fixed length."""
    wav, _ = librosa.load(path, sr=sr, mono=True)
    target = int(sr * duration)
    if len(wav) < target:
        wav = np.pad(wav, (0, target - len(wav)))
    else:
        start = (len(wav) - target) // 2
        wav = wav[start:start + target]
    return wav

def wav_to_melspec(wav):
    mel = librosa.feature.melspectrogram(
        y=wav, sr=CFG.sample_rate,
        n_mels=CFG.n_mels,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
        fmin=CFG.fmin, fmax=CFG.fmax
    )
    logmel = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
    logmel = (logmel - logmel.min()) / (logmel.max() - logmel.min() + 1e-6)

    # ✅ NEW: delta features
    delta  = librosa.feature.delta(logmel)
    delta2 = librosa.feature.delta(logmel, order=2)

    spec = np.stack([logmel, delta, delta2], axis=0)  # (3, H, W)

    return spec


In [4]:

# ── 3b. PRECOMPUTE SPECS TO DISK (run once — saves ~5h of repeated librosa calls)
def precompute_specs():
    os.makedirs(CFG.train_spec_dir, exist_ok=True)
    os.makedirs(CFG.test_spec_dir,  exist_ok=True)

    train_df = pd.read_csv(CFG.train_csv)
    print(f"Precomputing {len(train_df)} train specs...")
    for i, row in train_df.iterrows():
        out = f"{CFG.train_spec_dir}/{row['id']}.npy"
        if not os.path.exists(out):
            wav  = load_audio(os.path.join(CFG.train_dir, f"{row['id']}.wav"))
            spec = wav_to_melspec(wav)
            np.save(out, spec)
        if i % 5000 == 0:
            print(f"  train {i}/{len(train_df)}")

    test_ids = [f.replace(".wav", "") for f in os.listdir(CFG.test_dir) if f.endswith(".wav")]
    print(f"Precomputing {len(test_ids)} test specs...")
    for i, id_ in enumerate(test_ids):
        out = f"{CFG.test_spec_dir}/{id_}.npy"
        if not os.path.exists(out):
            wav  = load_audio(os.path.join(CFG.test_dir, f"{id_}.wav"))
            spec = wav_to_melspec(wav)
            np.save(out, spec)
        if i % 5000 == 0:
            print(f"  test {i}/{len(test_ids)}")

    print("✅ Precompute done")

precompute_specs()

Precomputing 37800 train specs...
  train 0/37800
  train 5000/37800
  train 10000/37800
  train 15000/37800
  train 20000/37800
  train 25000/37800
  train 30000/37800
  train 35000/37800
Precomputing 16200 test specs...
  test 0/16200
  test 5000/16200
  test 10000/16200
  test 15000/16200
✅ Precompute done


In [5]:
# ── 4. AUGMENTATION ──────────────────────────────────────────
def augment_wav(wav, sr=CFG.sample_rate):
    """Randomly apply one or more augmentations to a waveform."""
    # Additive white noise
    if random.random() < 0.5:
        noise = np.random.randn(len(wav)).astype(np.float32)
        wav = wav + noise * random.uniform(0.002, 0.008)

    # Time shift (circular)
    if random.random() < 0.5:
        shift = random.randint(-int(sr * 0.1), int(sr * 0.1))
        wav = np.roll(wav, shift)

    return wav.astype(np.float32)

def spec_augment(spec):
    """SpecAugment: frequency & time masking on the spectrogram."""
    spec = spec.copy()
    n_mels, time = spec.shape

    # Frequency masking
    for _ in range(2):
        f = random.randint(0, 16) # increased - mod3
        f0 = random.randint(0, n_mels - f)
        spec[f0:f0+f, :] = 0.0

    # Time masking
    for _ in range(2):
        t = random.randint(0, 30) # increased - mod4
        t0 = random.randint(0, time - t)
        spec[:, t0:t0+t] = 0.0

    return spec

In [6]:

# ── 5. DATASET (loads precomputed .npy — no librosa at runtime) ───────────────
class AudioDataset(Dataset):
    def __init__(self, df, audio_dir, augment=False, spec_dir=None):
        self.df        = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.augment   = augment
        self.spec_dir  = spec_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load precomputed spec if available
        if self.spec_dir and os.path.exists(f"{self.spec_dir}/{row['id']}.npy"):
            spec = np.load(f"{self.spec_dir}/{row['id']}.npy")  # (3, H, W)
        else:
            path = os.path.join(self.audio_dir, f"{row['id']}.wav")
            wav  = load_audio(path)
            spec = wav_to_melspec(wav)  # (3, H, W)

        # SpecAugment (works on all channels)
        if self.augment:
            for c in range(spec.shape[0]):
                spec[c] = spec_augment(spec[c])

        tensor = torch.tensor(spec, dtype=torch.float32)

        label = int(row['label']) if 'label' in row else -1
        return tensor, label

In [7]:
# ── 6. MODEL ─────────────────────────────────────────────────
class DigitCNN(nn.Module):
    """ResNet-18 backbone with a custom head for 10-class digit recognition."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = resnet18(weights=None)   # weights=None replaces deprecated pretrained=False
        self.backbone.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1,
                                        padding=1, bias=False)
        self.backbone.maxpool = nn.Identity()
        in_feats = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.2), # decreased - mod4
            nn.Linear(in_feats, 256),
            nn.ReLU(),
            nn.Dropout(0.1), # decreased - mod4
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

In [8]:

# ── 7. TRAINING UTILITIES ────────────────────────────────────

def mixup(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0)).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam

def train_one_epoch(model, loader, optimizer, scaler, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1) #increased - mod4

    for specs, labels in loader:
        specs, labels = specs.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=CFG.mixed_prec and device == "cuda"):
            # applying mixup (mod2)
            if random.random() < 0.5:
                specs, y_a, y_b, lam = mixup(specs, labels)
            
                logits = model(specs)
                loss = lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)
            else:
                logits = model(specs)
                loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)

    return total_loss / total, correct / total

@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0, 0, 0

    for specs, labels in loader:
        specs, labels = specs.to(device), labels.to(device)
        logits = model(specs)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)

    return total_loss / total, correct / total

In [9]:

# ── 8. FULL TRAINING WITH K-FOLD ─────────────────────────────
def run_training():
    train_df = pd.read_csv(CFG.train_csv)
    test_ids = [f.replace(".wav", "") for f in os.listdir(CFG.test_dir) if f.endswith(".wav")]
    test_df  = pd.DataFrame({"id": sorted(test_ids)})
    print(f"Test samples found: {len(test_df)}")

    skf    = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    device = CFG.device
    print(f"Training on: {device}")

    oof_preds  = np.zeros((len(train_df), 10))
    test_preds = np.zeros((len(test_df),  10))

    # Build test loader once (no augment, uses precomputed specs)
    test_ds = AudioDataset(test_df, CFG.test_dir, augment=False,
                           spec_dir=CFG.test_spec_dir)
    test_loader = DataLoader(test_ds, batch_size=CFG.batch_size * 2,
                             shuffle=False, num_workers=CFG.num_workers,
                             pin_memory=True)

    for fold, (train_idx, val_idx) in enumerate(
            skf.split(train_df, train_df['label'])):

        print(f"\n{'='*50}")
        print(f"  FOLD {fold+1} / {CFG.n_folds}")
        print(f"{'='*50}")

        tr_df  = train_df.iloc[train_idx]
        val_df = train_df.iloc[val_idx]

        # Use precomputed specs for both train and val
        tr_ds  = AudioDataset(tr_df,  CFG.train_dir, augment=True,
                              spec_dir=CFG.train_spec_dir)
        val_ds = AudioDataset(val_df, CFG.train_dir, augment=False,
                              spec_dir=CFG.train_spec_dir)

        tr_loader  = DataLoader(tr_ds,  batch_size=CFG.batch_size,
                                shuffle=True,  num_workers=CFG.num_workers,
                                pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=CFG.batch_size * 2,
                                shuffle=False, num_workers=CFG.num_workers,
                                pin_memory=True)

        model     = DigitCNN(num_classes=10).to(device)
        optimizer = torch.optim.AdamW(model.parameters(),
                                      lr=CFG.lr, weight_decay=CFG.weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                        optimizer, T_max=CFG.epochs, eta_min=1e-6)
        scaler = torch.cuda.amp.GradScaler(enabled=CFG.mixed_prec and device == "cuda")

        best_val_acc = 0.0
        ckpt_path    = f"fold{fold+1}_best.pt"

        for epoch in range(1, CFG.epochs + 1):
            tr_loss, tr_acc   = train_one_epoch(model, tr_loader,
                                                optimizer, scaler, device)
            val_loss, val_acc = validate(model, val_loader, device)
            scheduler.step()

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), ckpt_path)

            print(f"  Epoch {epoch:02d}  "
                  f"tr_loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}  "
                  f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}"
                  + ("  ★" if val_acc == best_val_acc else ""))

        # ── OOF predictions (fixed indexing) ──────────────────
        model.load_state_dict(torch.load(ckpt_path, map_location=device,
                                         weights_only=True))
        model.eval()
        all_probs = []
        with torch.no_grad():
            for specs, _ in val_loader:
                p = F.softmax(model(specs.to(device)), dim=1).cpu().numpy()
                all_probs.append(p)
        oof_preds[val_idx] = np.vstack(all_probs)   # ← correct indexing

        # ── Test predictions (TTA × tta_rounds) ───────────────
        fold_test_preds = np.zeros((len(test_df), 10))
        for _ in range(CFG.tta_rounds):
            tta_ds = AudioDataset(test_df, CFG.test_dir, augment=True,
                                  spec_dir=CFG.test_spec_dir)
            tta_loader = DataLoader(tta_ds, batch_size=CFG.batch_size * 2,
                                    shuffle=False, num_workers=CFG.num_workers,
                                    pin_memory=True)
            batch_probs = []
            with torch.no_grad():
                for specs, _ in tta_loader:
                    p = F.softmax(model(specs.to(device)), dim=1).cpu().numpy()
                    batch_probs.append(p)
            if batch_probs:
                fold_test_preds += np.vstack(batch_probs) / CFG.tta_rounds

        test_preds += fold_test_preds / CFG.n_folds

        print(f"\n  Best val accuracy fold {fold+1}: {best_val_acc:.4f}")

    # ── OOF accuracy ──────────────────────────────────────────
    oof_labels = train_df['label'].values
    oof_acc    = (oof_preds.argmax(1) == oof_labels).mean()
    print(f"\n{'='*50}")
    print(f"  OOF Accuracy (all folds): {oof_acc:.4f}")
    print(f"{'='*50}\n")

    # ── Submission ────────────────────────────────────────────
    sub = test_df.copy()
    sub['label'] = test_preds.argmax(1)
    sub[['id', 'label']].to_csv("submission.csv", index=False)
    print("submission.csv saved ✔")
    return sub



In [10]:
# ── 9. RUN ───────────────────────────────────────────────────
if __name__ == "__main__":
    submission = run_training()
    print(submission.head(10))

Test samples found: 16200
Training on: cuda

  FOLD 1 / 5
  Epoch 01  tr_loss=1.5420  tr_acc=0.4875  val_loss=1.2541  val_acc=0.5857  ★
  Epoch 02  tr_loss=1.0111  tr_acc=0.7189  val_loss=0.3089  val_acc=0.9599  ★
  Epoch 03  tr_loss=0.8572  tr_acc=0.7495  val_loss=0.3185  val_acc=0.9398
  Epoch 04  tr_loss=0.8420  tr_acc=0.7264  val_loss=0.2228  val_acc=0.9642  ★
  Epoch 05  tr_loss=0.8459  tr_acc=0.7060  val_loss=0.2919  val_acc=0.9800  ★
  Epoch 06  tr_loss=0.8404  tr_acc=0.6921  val_loss=0.2448  val_acc=0.9835  ★
  Epoch 07  tr_loss=0.7557  tr_acc=0.7530  val_loss=0.1874  val_acc=0.9788
  Epoch 08  tr_loss=0.7543  tr_acc=0.8025  val_loss=0.1535  val_acc=0.9918  ★
  Epoch 09  tr_loss=0.7795  tr_acc=0.7637  val_loss=0.2312  val_acc=0.9865
  Epoch 10  tr_loss=0.8108  tr_acc=0.7599  val_loss=0.1752  val_acc=0.9878
  Epoch 11  tr_loss=0.7970  tr_acc=0.7433  val_loss=0.2266  val_acc=0.9771
  Epoch 12  tr_loss=0.7933  tr_acc=0.7478  val_loss=0.1781  val_acc=0.9886
  Epoch 13  tr_loss=0.73